In [2]:
# Cài đặt các thư viện thiếu
!pip install ultralytics roboflow pandas matplotlib seaborn

import os
import yaml
import torch
import gc
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from roboflow import Roboflow
from ultralytics import YOLO

def clean_gpu():
    torch.cuda.empty_cache()
    gc.collect()

# Danh sách 5 mô hình tiêu chuẩn
standard_models = [
    {"name": "1_YOLOv8n", "weights": "yolov8n.pt"},
    {"name": "2_YOLOv10n", "weights": "yolov10n.pt"},
    {"name": "3_YOLOv11n", "weights": "yolo11n.pt"},
    {"name": "4_YOLOv9c", "weights": "yolov9c.pt"},
    {"name": "5_RTDETR_Hybrid", "weights": "rtdetr-l.pt"}
]

print("✅ Thư viện đã sẵn sàng!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 113.4 MB/s eta 0:00:0000:01
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source 

In [3]:
# 1. Tải Dataset 2
rf2 = Roboflow(api_key="zau700BeI9He3qiUGWgF")
project2 = rf2.workspace("yolo-i9xsa").project("car-traffic-utuxv")
ds2 = project2.version(1).download("yolov8")

# 2. Tiền xử lý: Gộp Car/Truck về 0, Xóa nhãn Stop (ID=2)
for lb in glob(f"{ds2.location}/*/labels/*.txt"):
    with open(lb, 'r') as f: lines = f.readlines()
    # Chỉ giữ ID 0, 1 và đổi tất cả thành 0
    new_l = ["0 " + " ".join(l.split()[1:]) + "\n" for l in lines if l.split()[0] in ['0', '1']]
    with open(lb, 'w') as f: f.writelines(new_l)

# 3. Cập nhật data.yaml
with open(f"{ds2.location}/data.yaml", 'r') as f: cfg2 = yaml.safe_load(f)
cfg2['names'], cfg2['nc'] = {0: 'vehicle'}, 1
with open(f"{ds2.location}/data.yaml", 'w') as f: yaml.dump(cfg2, f)

# 4. Huấn luyện chuỗi 5 mô hình
for m in standard_models:
    print(f"🚀 DS2 - Đang train: {m['name']}")
    model = YOLO(m['weights'])
    model.train(data=f"{ds2.location}/data.yaml", epochs=30, imgsz=640, name=f"DS2_{m['name']}", exist_ok=True)
    clean_gpu()

# 5. Đóng gói kết quả Bộ 2
shutil.make_archive('Results_Dataset_2', 'zip', 'runs/detect')
print("✅ Đã hoàn thành và đóng gói Bộ 2!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to car-traffic-1 in yolov8:: 100%|██████████| 1913/1913 [00:00<00:00, 8382.48it/s]


🚀 DS2 - Đang train: 1_YOLOv8n
Ultralytics 8.4.49 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/car-traffic-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=DS2_1_YOLOv8n, nbs=64, nms=False, opset=None, optimize=False, o

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       1/30      12.6G     0.5167      1.047     0.1415         86        640: 100% ━━━━━━━━━━━━ 42/42 1.4s/it 57.4s1.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.2s/it 7.4s0.8ss
                   all        191        964       0.86      0.766      0.878      0.715

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

      28/30      12.4G     0.1254     0.2731    0.02523         61        640: 100% ━━━━━━━━━━━━ 42/42 1.1s/it 45.7s1.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.7it/s 3.6s0.7ss
                   all        191        964      0.896      0.908      0.955      0.858

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      29/30      12.3G     0.1264     0.2733    0.02391         57        640: 100% ━━━━━━━━━━━━ 42/42 1.1s/it 45.8s1.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.7it/s 3.6s0.7ss
                   all        191        964      0.881      0.912      0.958      0.859

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      30/30      12.4G     0.1243     0.2651    0.02485         62        640: 100% ━━━━━━━━━━━━ 42/42 1.1s/it 45.7s1.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.7it/s 3.6s0.7ss
                   all        191        964      0.898      0.893      0.955      0.857

30 epochs completed in 0.432 hours.
Optimizer stripped from /kaggle/working/runs/detect/DS2_5_RTDETR_Hybrid/weights/last.pt, 66.2MB
Optimizer stripped from /kaggle/working/runs/detect/DS2_5_RTDETR_Hybrid/weights/best.pt, 66.2MB

Validating /kaggle/working/runs/detect/DS2_5_RTDETR_Hybrid/weights/best.pt...
Ultralytics 8.4.49 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.3it/s 4.7s0.9ss
                   all        191        964      0.881   

In [4]:
# 1. Tải Dataset 3
rf3 = Roboflow(api_key="zau700BeI9He3qiUGWgF")
project3 = rf3.workspace("project-tdxxb").project("cctv_car_bike_detection-fhqk8")
ds3 = project3.version(3).download("yolov8")

# 2. Tiền xử lý: Ép tất cả class hiện có (Car, Motorbike) về ID 0
for lb in glob(f"{ds3.location}/*/labels/*.txt"):
    with open(lb, 'r') as f: lines = f.readlines()
    new_l = ["0 " + " ".join(l.split()[1:]) + "\n" for l in lines if len(l.split()) > 0]
    with open(lb, 'w') as f: f.writelines(new_l)

# 3. Cập nhật data.yaml
with open(f"{ds3.location}/data.yaml", 'r') as f: cfg3 = yaml.safe_load(f)
cfg3['names'], cfg3['nc'] = {0: 'vehicle'}, 1
with open(f"{ds3.location}/data.yaml", 'w') as f: yaml.dump(cfg3, f)

# 4. Huấn luyện chuỗi 5 mô hình (Tiếp tục trong cùng thư mục runs)
for m in standard_models:
    print(f"🚀 DS3 - Đang train: {m['name']}")
    model = YOLO(m['weights'])
    model.train(data=f"{ds3.location}/data.yaml", epochs=30, imgsz=640, name=f"DS3_{m['name']}", exist_ok=True)
    clean_gpu()

# 5. Đóng gói toàn bộ kết quả (Cả DS2 và DS3 nếu bạn chưa xóa thư mục runs)
shutil.make_archive('Full_Project_Results', 'zip', 'runs/detect')
print("✅ Đã hoàn thành tất cả!")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to cctv_car_bike_detection-3 in yolov8:: 100%|██████████| 270/270 [00:00<00:00, 6991.63it/s]


🚀 DS3 - Đang train: 1_YOLOv8n
Ultralytics 8.4.49 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/cctv_car_bike_detection-3/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=DS3_1_YOLOv8n, nbs=64, nms=False, opset=None, optim

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       1/30      11.8G      1.652     0.8031     0.5643        458        640: 100% ━━━━━━━━━━━━ 6/6 1.2s/it 7.3s1.4ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.7it/s 0.7s2.2s
                   all         33        792      0.033      0.136     0.0126    0.00486

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


...

      28/30      11.8G     0.5024     0.4523    0.09158        276        640: 100% ━━━━━━━━━━━━ 6/6 1.1s/it 6.4s1.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.8it/s 0.7s2.2s
                   all         33        792      0.762      0.852      0.854      0.407

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      29/30      12.5G     0.5017     0.4449    0.09249        248        640: 100% ━━━━━━━━━━━━ 6/6 1.1s/it 6.3s1.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.8it/s 0.7s2.2s
                   all         33        792       0.76      0.861      0.857      0.409

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      30/30      12.5G     0.4897     0.4415     0.0917        264        640: 100% ━━━━━━━━━━━━ 6/6 1.1s/it 6.4s1.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.8it/s 0.7s2.2s
                   all         33        792       0.77      0.863      0.872      0.411

30 epochs completed in 0.078 hours.
Optimizer stripped from /kaggle/working/runs/detect/DS3_5_RTDETR_Hybrid/weights/last.pt, 66.2MB
Optimizer stripped from /kaggle/working/runs/detect/DS3_5_RTDETR_Hybrid/weights/best.pt, 66.2MB

Validating /kaggle/working/runs/detect/DS3_5_RTDETR_Hybrid/weights/best.pt...
Ultralytics 8.4.49 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.9it/s 0.7s2.1s
                   all         33        792       0.77      0.